In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q18/q18_rewrite/checkpoints/pre_cell_3.pickle

In [ ]:
%%cudf.pandas.profile
### cell 3 ###

# Optimized cell 3
gb1 = lineitem.groupby("L_ORDERKEY", as_index=False, sort=False)["L_QUANTITY"].sum()
# filter orders with total quantity > 300
gb1 = gb1.loc[gb1.L_QUANTITY > 300]
# project only needed columns from orders and customer to reduce data movement
orders_sub   = orders[["O_ORDERKEY", "O_CUSTKEY", "O_ORDERDATE", "O_TOTALPRICE"]]
customer_sub = customer[["C_CUSTKEY", "C_NAME"]]
# join and sort, avoiding a second groupby
total = (
    gb1
      .merge(orders_sub,   left_on="L_ORDERKEY", right_on="O_ORDERKEY")
      .merge(customer_sub, left_on="O_CUSTKEY", right_on="C_CUSTKEY")
      [["C_NAME", "C_CUSTKEY", "O_ORDERKEY", "O_ORDERDATE", "O_TOTALPRICE", "L_QUANTITY"]]
      .sort_values(["O_TOTALPRICE", "O_ORDERDATE"], ascending=[False, True])
)